# Exercise: Plotly Quickstart con Datos de Incautación de Estupefacientes

**Objetivo:** construir tres visualizaciones interactivas antes de pasar al dashboard completo.

**Dataset:** `Incautación_de_Estupefacientes__20260423.csv`


## Parte 1. Carga y entiende el dataset

Antes de graficar, identifica:

1. qué representa cada fila
2. qué filtros serían útiles
3. qué métrica numérica será el centro del dashboard


In [3]:
from pathlib import Path

import pandas as pd
import plotly.express as px

# First, find the correct path to the CSV file
import subprocess
result = subprocess.run(['find', '/workspaces/Mineria_Datos', '-name', '*Incautacion*', '-o', '-name', '*Estupefacientes*'], 
                       capture_output=True, text=True)
print("Found files:")
print(result.stdout)

# Update this path based on the search results above
DATA_PATH = Path('/workspaces/Mineria_Datos/path/to/your/file/Incautación_de_Estupefacientes__20260423.csv')

df = pd.read_csv('/workspaces/Mineria_Datos/analitica-datos-estudiantes-main/semana11/Incautación_de_Estupefacientes._20260423.csv', low_memory=False)
df = df.rename(
    columns={
        'DEPARTAMENTO': 'departamento',
        'MUNICIPIO': 'municipio',
        'CODIGO DANE': 'codigo_dane',
        'CLASE BIEN': 'sustancia',
        'FECHA HECHO': 'fecha',
        'CANTIDAD': 'cantidad',
    }
)

df['fecha'] = pd.to_datetime(df['fecha'], dayfirst=True, errors='coerce')
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')
df = df.dropna(subset=['fecha', 'cantidad']).sort_values('fecha')
df['anio'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.to_period('M').astype(str)

df.head()


Found files:
/workspaces/Mineria_Datos/analitica-datos-estudiantes-main/semana11/Incautación_de_Estupefacientes._20260423.csv



,departamento,municipio,codigo_dane,sustancia,fecha,cantidad,anio,mes
0,BOLÍVAR,Cartagena (CT),13001000,MARIHUANA,2010-01-01,28.0,2010,2010-01
87,QUINDÍO,Armenia (CT),63001000,COCAINA,2010-01-01,2.0,2010,2010-01
86,SANTANDER,Bucaramanga (CT),68001000,BASUCO,2010-01-01,20.0,2010,2010-01
85,ANTIOQUIA,Bello,05088000,MARIHUANA,2010-01-01,10.0,2010,2010-01
84,CUNDINAMARCA,Puerto Salgar,25572000,MARIHUANA,2010-01-01,20.0,2010,2010-01


In [4]:
df.describe(include='all')


,departamento,municipio,codigo_dane,sustancia,fecha,cantidad,anio,mes
count,1953250,1953240,1953282,1953286,1953286,1.953286e+06,1.953286e+06,1953286
unique,39,1003,2731,5,NaN,NaN,NaN,195
top,ANTIOQUIA,Bogotá D.C. (CT),11001000,MARIHUANA,NaN,NaN,NaN,2016-02
freq,496195,168067,168061,964905,NaN,NaN,NaN,23079
mean,NaN,NaN,NaN,NaN,2016-02-20 12:10:00.111197,4.770425e+03,2.015700e+03,NaN
min,NaN,NaN,NaN,NaN,2010-01-01 00:00:00,0.000000e+00,2.010000e+03,NaN
25%,NaN,NaN,NaN,NaN,2013-02-17 00:00:00,6.000000e+00,2.013000e+03,NaN
50%,NaN,NaN,NaN,NaN,2015-05-23 00:00:00,1.800000e+01,2.015000e+03,NaN
75%,NaN,NaN,NaN,NaN,2018-03-17 00:00:00,6.000000e+01,2.018000e+03,NaN
max,NaN,NaN,NaN,NaN,2026-03-31 00:00:00,2.500000e+07,2.026000e+03,NaN


## Parte 2. Tendencia temporal

Crea un gráfico de línea con la cantidad total incautada por año y color por tipo de sustancia.


In [5]:
trend_df = (
    df.groupby(['anio', 'sustancia'], as_index=False)['cantidad']
    .sum()
    .sort_values('anio')
)

fig = px.line(
    trend_df,
    x='anio',
    y='cantidad',
    color='sustancia',
    markers=True,
    title='Cantidad total incautada por año y sustancia'
)
fig.update_layout(xaxis_title='Año', yaxis_title='Cantidad (g/kg/unidades)')
fig.show()


## Parte 3. Comparación entre departamentos

Crea un gráfico de barras con la cantidad total incautada por departamento (top 15).


In [6]:
depto_df = (
    df.groupby('departamento', as_index=False)['cantidad']
    .sum()
    .sort_values('cantidad', ascending=False)
    .head(15)
)

fig = px.bar(
    depto_df,
    x='departamento',
    y='cantidad',
    color='departamento',
    title='Top 15 departamentos por cantidad incautada'
)
fig.update_layout(
    xaxis_title='Departamento',
    yaxis_title='Cantidad total',
    xaxis_tickangle=-45,
    showlegend=False
)
fig.show()


## Parte 4. Distribución por sustancia

Crea un box plot que muestre cómo se distribuyen las cantidades según tipo de sustancia.


In [ ]:
# Filtrar outliers extremos para mejor visualización
q99 = df['cantidad'].quantile(0.99)
df_filtered = df[df['cantidad'] <= q99]

fig = px.box(
    df_filtered,
    x='sustancia',
    y='cantidad',
    color='sustancia',
    title='Distribución de cantidades incautadas por sustancia (sin outliers extremos)'
)
fig.update_layout(
    xaxis_title='Sustancia',
    yaxis_title='Cantidad',
    showlegend=False
)
fig.show()


## Parte 5 (Bonus). Mapa de calor mensual

Visualiza cómo evoluciona la incautación mes a mes para detectar estacionalidad.


In [ ]:
heatmap_df = (
    df.groupby(['anio', 'sustancia'], as_index=False)['cantidad']
    .sum()
)

fig = px.density_heatmap(
    heatmap_df,
    x='anio',
    y='sustancia',
    z='cantidad',
    color_continuous_scale='Oranges',
    title='Mapa de calor: cantidad incautada por año y sustancia'
)
fig.update_layout(xaxis_title='Año', yaxis_title='Sustancia')
fig.show()


## Cierre

Responde en una celda markdown:

- ¿qué filtro pondrías primero en una app?
- ¿qué KPI tendría más sentido arriba del dashboard?
- ¿qué insight aparece más rápido con estos tres gráficos?


Escribe aquí tu reflexión final.
